# L3: Multi-agent Customer Support Automation

In this lesson, you will learn about the six key elements which help make Agents perform even better:
- Role Playing
- Focus
- Tools
- Cooperation
- Guardrails
- Memory

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [ ]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import libraries, API and LLM

In [1]:
from crewai import Agent, Task, Crew

In [2]:
import os
from com.example.agentic.utils import get_openai_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'llama3.2:1b-instruct-q8_0'

from crewai import LLM
llm = LLM(model="ollama/llama3.2:1b-instruct-q8_0", base_url="http://localhost:11434")

## Role Playing, Focus and Cooperation

In [3]:
support_agent = Agent(
    role="Senior Support Representative",
	goal="Be the most friendly and helpful "
        "support representative in your team",
	backstory=(
		"You work at crewAI (https://crewai.com) and "
        "are now working on providing "
		"support to {customer}, a super important customer "
        "for your company."
		"You need to make sure that you provide the best support!"
		"Make sure to provide full complete answers, "
        " and make no assumptions."
	),
    allow_delegation=False,
	verbose=True,
    llm=llm
)

- By not setting `allow_delegation=False`, `allow_delegation` takes its default value of being `True`.
- This means the agent _can_ delegate its work to another agent which is better suited to do a particular task. 

In [4]:
support_quality_assurance_agent = Agent(
	role="Support Quality Assurance Specialist",
	goal="Get recognition for providing the "
    "best support quality assurance in your team",
	backstory=(
		"You work at crewAI (https://crewai.com) and "
        "are now working with your team "
		"on a request from {customer} ensuring that "
        "the support representative is "
		"providing the best support possible.\n"
		"You need to make sure that the support representative "
        "is providing full"
		"complete answers, and make no assumptions."
	),
	verbose=True,
    llm=llm
)

* **Role Playing**: Both agents have been given a role, goal and backstory.
* **Focus**: Both agents have been prompted to get into the character of the roles they are playing.
* **Cooperation**: Support Quality Assurance Agent can delegate work back to the Support Agent, allowing for these agents to work together.

## Tools, Guardrails and Memory

### Tools

- Import CrewAI tools

In [5]:
from crewai_tools import SerperDevTool, \
                         ScrapeWebsiteTool, \
                         WebsiteSearchTool

### Possible Custom Tools
- Load customer data
- Tap into previous conversations
- Load data from a CRM
- Checking existing bug reports
- Checking existing feature requests
- Checking ongoing tickets
- ... and more

- Some ways of using CrewAI tools.

```Python
search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()
```

- Instantiate a document scraper tool.
- The tool will scrape a page (only 1 URL) of the CrewAI documentation.

In [6]:
from com.example.agentic.tools.config import _tool_config


docs_scrape_tool = ScrapeWebsiteTool(
    config=_tool_config,
    website_url="https://docs.crewai.com/how-to/Creating-a-Crew-and-kick-it-off/"
)

USER_AGENT environment variable not set, consider setting it to identify your requests.


##### Different Ways to Give Agents Tools

- Agent Level: The Agent can use the Tool(s) on any Task it performs.
- Task Level: The Agent will only use the Tool(s) when performing that specific Task.

**Note**: Task Tools override the Agent Tools.

### Creating Tasks
- You are passing the Tool on the Task Level.

In [7]:
inquiry_resolution = Task(
    description=(
        "{customer} just reached out with a super important ask:\n"
	    "{inquiry}\n\n"
        "{person} from {customer} is the one that reached out. "
		"Make sure to use everything you know "
        "to provide the best support possible."
		"You must strive to provide a complete "
        "and accurate response to the customer's inquiry."
    ),
    expected_output=(
	    "A detailed, informative response to the "
        "customer's inquiry that addresses "
        "all aspects of their question.\n"
        "The response should include references "
        "to everything you used to find the answer, "
        "including external data or solutions. "
        "Ensure the answer is complete, "
		"leaving no questions unanswered, and maintain a helpful and friendly "
		"tone throughout."
    ),
	tools=[docs_scrape_tool],
    agent=support_agent,
)

- `quality_assurance_review` is not using any Tool(s)
- Here the QA Agent will only review the work of the Support Agent

In [8]:
quality_assurance_review = Task(
    description=(
        "Review the response drafted by the Senior Support Representative for {customer}'s inquiry. "
        "Ensure that the answer is comprehensive, accurate, and adheres to the "
		"high-quality standards expected for customer support.\n"
        "Verify that all parts of the customer's inquiry "
        "have been addressed "
		"thoroughly, with a helpful and friendly tone.\n"
        "Check for references and sources used to "
        " find the information, "
		"ensuring the response is well-supported and "
        "leaves no questions unanswered."
    ),
    expected_output=(
        "A final, detailed, and informative response "
        "ready to be sent to the customer.\n"
        "This response should fully address the "
        "customer's inquiry, incorporating all "
		"relevant feedback and improvements.\n"
		"Don't be too formal, we are a chill and cool company "
	    "but maintain a professional and friendly tone throughout."
    ),
    agent=support_quality_assurance_agent,
)


### Creating the Crew

#### Memory
- Setting `memory=True` when putting the crew together enables Memory.

In [9]:
from crewai import Crew, Agent, Task, Process, Memory
from com.example.agentic.tools.config import _embedder_config_ollama
from crewai.rag.embeddings.factory import build_embedder

memory = Memory(
    llm=llm,
    embedder={
    "provider": "ollama",
    "config": {
        "model_name": "nomic-embed-text",
        "url": "http://localhost:11434/api/embeddings"
    },
})

crew = Crew(
    agents=[support_agent, support_quality_assurance_agent],
    tasks=[inquiry_resolution, quality_assurance_review],
    process=Process.sequential,
    verbose=True,
    memory=memory,
    llm=llm
)

### Running the Crew

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

#### Guardrails
- By running the execution below, you can see that the agents and the responses are within the scope of what we expect from them.

In [10]:
inputs = {
    "customer": "DeepLearningAI",
    "person": "Andrew Ng",
    "inquiry": "I need help with setting up a Crew "
               "and kicking it off, specifically "
               "how can I add memory to my crew? "
               "Can you provide guidance?"
}
result = crew.kickoff(inputs=inputs)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: e35a5131-ac37-4ec8-9ec9-c8e83283efa5                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Invalid type Memory for attribute 'crew_memory' value. Expected one of ['bool', 'str', 'bytes', 'int', 'float'] or a sequence of those types


╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: DeepLearningAI just reached out with a super important ask:                                              │
│  I need help with setting up a Crew and kicking it off, specifically how can I add memory to my crew? Can you   │
│  provide guidance?                                                                                              │
│                                                                                                                 │
│  Andrew Ng from DeepLearningAI is the one that reached out. Make sure to use everything you know to provide     │
│  the best support possible.You must strive to provide a complete and accurate response to the customer's        │
│  inquiry.                                                                                                       │
│  ID: 48f8e022-4fe0-4393-af2e-21e650df715e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 🧠 Memory Retrieval ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Retrieval Started                                                                                       │
│  Status: Retrieving...                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 🧠 Memory Retrieved ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Retrieval Completed                                                                                     │
│  Time: 6689.71ms                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Review the response drafted by the Senior Support Representative for DeepLearningAI's inquiry. Ensure    │
│  that the answer is comprehensive, accurate, and adheres to the high-quality standards expected for customer    │
│  support.                                                                                                       │
│  Verify that all parts of the customer's inquiry have been addressed thoroughly, with a helpful and friendly    │
│  tone.                                                                                                          │
│  Check for references and sources used to  find the information, ensuring the response is well-supported and    │
│  leaves no questions unanswered.                                                                                │
│  Agent: Support Quality Assurance Specialist                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🧠 Memory Save ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Save Started                                                                                            │
│  Status: Saving...                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: e35a5131-ac37-4ec8-9ec9-c8e83283efa5                                                                       │
│  Final Output: {                                                                                                │
│  "name": "crew",                                                                                                │
│  "parameters": {                                                                                                │
│  "+addmemory": "{\"type\":\"array\",\"items\":{\"type\":\"string\"}}"                                           │
│  }                                                                                                              │
│  {                                                                                                              │
│  "operation": "search_memory",                                                                                  │
│  "parameters": {                                                                                                │
│  "+addquery": "{\"type\":\"object\",\"description\": \"Pass one or more queries to search for multiple things   │
│  at once. Use this when you need to find facts, decisions, preferences, or past results that may have been      │
│  stored                                                                                                         │
│  previously.\",\"members\":{\"type\":\"array\",\"required\",\"minimal[0]\":1,\"max[0]\":2,\"items\":{\"type\":  │
│  \"string\"}},\"default\":[],\"required\":[],\"example\":[\"I want to learn more about the Crew feature\"]}}"   │
│  }                                                                                                              │
│  {                                                                                                              │
│  "operation": "search_memory",                                                                                  │
│  "parameters": {                                                                                                │
│  "+addquery": "{\"type\":\"object\",\"description \": \"Pass a single item or multiple items at                 │
│  once.\",\"members\":{\"type\":\"array\",\"required\",\"minimal[0]\":1,\"max[0]\":2,\"items\":{\"type\":\"stri  │
│  ng\"}},\"default\":[],\"required\":[],\"example\":[\"I want to learn more about the Crew feature\"]},          │
│  +additem": "{\"type\":\"object\",\"description \": \"Pass one or more facts, decisions, observations, or       │
│  lessons to                                                                                                     │
│  remember.\",\"members\":{\"type\":\"array\",\"required\",\"minimal[0]\":1,\"max[0]\":2,\"items\":{\"type\":\"  │
│  string\"}},\"default\":[],\"required\":[],\"example\":[\"I have learned so much about the Crew feature\"]}}"   │
│  }                                                                                                              │
│  {                                                                                                              │
│  "operation": "save_to_memory",                                                                                 │
│  "parameters": {                                                                                                │
│  "+additem": "{\"type\":\"string\", \"description\": \\"I am learning about how to better assist our            │
│  customers.\"}}  # A new item should be added for this

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── ✅ Memory Saved ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Save Completed                                                                                          │
│  Source: Unified Memory                                                                                         │
│  Time: 19321.30ms                                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

- Display the final result as Markdown.

In [16]:
print(f"Raw Output: {result.raw}")

Raw Output: {
"name": "crew",
"parameters": {
"+addmemory": "{\"type\":\"array\",\"items\":{\"type\":\"string\"}}"
}
{
"operation": "search_memory",
"parameters": {
"+addquery": "{\"type\":\"object\",\"description\": \"Pass one or more queries to search for multiple things at once. Use this when you need to find facts, decisions, preferences, or past results that may have been stored previously.\",\"members\":{\"type\":\"array\",\"required\",\"minimal[0]\":1,\"max[0]\":2,\"items\":{\"type\":\"string\"}},\"default\":[],\"required\":[],\"example\":[\"I want to learn more about the Crew feature\"]}}"
}
{
"operation": "search_memory",
"parameters": {
"+addquery": "{\"type\":\"object\",\"description \": \"Pass a single item or multiple items at once.\",\"members\":{\"type\":\"array\",\"required\",\"minimal[0]\":1,\"max[0]\":2,\"items\":{\"type\":\"string\"}},\"default\":[],\"required\":[],\"example\":[\"I want to learn more about the Crew feature\"]}, +additem": "{\"type\":\"object\",\"desc

In [17]:
from IPython.display import Markdown
Markdown(result.raw)

{
"name": "crew",
"parameters": {
"+addmemory": "{\"type\":\"array\",\"items\":{\"type\":\"string\"}}"
}
{
"operation": "search_memory",
"parameters": {
"+addquery": "{\"type\":\"object\",\"description\": \"Pass one or more queries to search for multiple things at once. Use this when you need to find facts, decisions, preferences, or past results that may have been stored previously.\",\"members\":{\"type\":\"array\",\"required\",\"minimal[0]\":1,\"max[0]\":2,\"items\":{\"type\":\"string\"}},\"default\":[],\"required\":[],\"example\":[\"I want to learn more about the Crew feature\"]}}"
}
{
"operation": "search_memory",
"parameters": {
"+addquery": "{\"type\":\"object\",\"description \": \"Pass a single item or multiple items at once.\",\"members\":{\"type\":\"array\",\"required\",\"minimal[0]\":1,\"max[0]\":2,\"items\":{\"type\":\"string\"}},\"default\":[],\"required\":[],\"example\":[\"I want to learn more about the Crew feature\"]}, +additem": "{\"type\":\"object\",\"description \": \"Pass one or more facts, decisions, observations, or lessons to remember.\",\"members\":{\"type\":\"array\",\"required\",\"minimal[0]\":1,\"max[0]\":2,\"items\":{\"type\":\"string\"}},\"default\":[],\"required\":[],\"example\":[\"I have learned so much about the Crew feature\"]}}"
}
{
"operation": "save_to_memory",
"parameters": {
"+additem": "{\"type\":\"string\", \"description\": \\"I am learning about how to better assist our customers.\"}}  # A new item should be added for this purpose
}